In [1]:
import pandas as pd
import sqlalchemy as sa
import sql

pd.set_option('display.max_rows', 10)

connection_url = sa.engine.URL.create(
    drivername="postgresql+psycopg2",
    host="95.163.241.236",
    port=5432,
    database="simulative",
    username="student",
    password="qweasd963",
)
engine = sa.create_engine(connection_url)

%load_ext sql

# %config SqlMagic.displaylimit = 20
%config SqlMagic.displaycon = False
# %config SqlMagic.feedback = False
%config SqlMagic.autopandas = True

%sql engine

print(f"Pandas ver. {pd.__version__}: порог усечения строк уменьшен до 10")
print(f"SQLAlchemy ver. {sa.__version__}: подключение создано")
print(f"JupySQL ver. {sql.__version__}: подключен через SQLAlchemy Engine")

Pandas ver. 3.0.5
SQLAlchemy ver. 2.0.52 – подключение создано
JupySQL ver. 0.11.1 – подключен через SQLAlchemy Engine



---

[ссылка](https://t.me/c/3825910988/465) Вы можете создавать свои поля, например используя CASE WHEN или просто указывая значение. Например:

Эта команда выведет все поля таблицы с пользователями:
```sql
SELECT *
FROM users
```
Этот селект уже более сложный:
```sql
SELECT 
u.*,
case when user_name like ‘U%’ then 1 else 0 end as user_u_flg,
'Пользователи' as types
FROM users u
```
Обратите внимание мы присвоили таблице users алиас (псевдоним) u и сказали выведи мне все столбцы из таблицы u и добавь 2 столбца: user_u_flg (флаг где имя пользователя начинается с u) и types (где мы просто указали тип руками).

Для чего это нужно? В 80% случаев мы не делаем обычный SELECT, мы на ходу рассчитываем необходимые нам поля

---

```sql
DELETE FROM data.clients;
WHERE client_id = 99;
```
можно сделать rollback, так как не было комита

---

[ссылка](https://t.me/c/3825910988/491) Надо в трим оборачивать

Что такое трим?

---

[ссыла](https://t.me/c/3825910988/605) Кстати, маленький совет, если фильтруетесь по дате, то делайте ее динамической, то есть вместо `>= date’2026-03-01’` указывайте
```sql
>= date_trunc('month', CURRENT_DATE) - INTERVAL '3 month')::date
```
Тогда вам не придется возвращаться и менять дату, а она будет автоматически брать последние 3 месяца. \
Ну и также стоит подсветить, что в условии WHERE можно использовать CASE WHEN, например:
```sql
WHERE case when user_name like 'U%' then 1 else 0 end = 1
```
То есть по сути мы говорим: если имя пользователя начинается на U, тогда это 1, если нет, то 0, и дальше фильтруем, говоря, выведи только тех у кого флаг = 1. 

---

[ссылка](https://t.me/c/3825910988/705) 04.09 \
Что такое `NVL` и `ADDMONTHS`?

Как минимум почти в каждом своем скрипте я использую CASE WHEN и COALESCE или `NVL`. И даже в примерах выше я упоминал что case when можно использовать в условиях where, а вы дополняли, что его в том числе можно использовать в order by. 
Делитесь в каких случаях вы используете эти операторы.

Еще одна сегодняшняя тема скалярные функции при работе с датами и временем. Это тоже очень важная часть работы с данными. Например, чтобы не грузить в отчет миллионы строк данные нужно агрегировать, в том числе и до месяца.
Здесь на помощь приходит динамическая дата, это как раз еще один способ работы с датами, где мы удаляем последние два месяца и записываем их заново, но уже с обновленными днями. И чтобы не менять даты как раз можно использовать INTERVAL или `ADDMONTHS`. \
Это как раз та тема, которая позволит вам автоматизировать свою отчетность.

---

## Задача от 04.09 17:00

:::{tip} Задача от 04.09 17:00
[ссылка](https://t.me/c/3825910988/758) \
Нужно перевернуть индекс, чтобы ccode остался в прежнем порядке. 
```sql
with cte as (
    select 1 as id, 'A' as ccode
    union all 
    select 2 as id, 'B' as ccode
    union all 
    select 3 as id, 'C' as ccode
    union all 
    select 4 as id, 'D' as ccode
    union all 
    select 5 as id, 'E' as ccode
)
select * from cte
```
:::

:::{note} Решения:
```sql
WITH cte AS (
    SELECT 1 AS id, 'A' AS ccode
    UNION ALL SELECT 2, 'B'
    UNION ALL SELECT 3, 'C'
    UNION ALL SELECT 4, 'D'
    UNION ALL SELECT 5, 'E'
)
SELECT 
    (SELECT MAX(id) FROM cte) + 1 - id AS id,
    ccode
FROM cte
ORDER BY ccode;
```
```sql
with cte as (
    select 1 as id, 'A' as ccode
    union all 
    select 2 as id, 'B' as ccode
    union all 
    select 3 as id, 'C' as ccode
    union all 
    select 4 as id, 'D' as ccode
    union all 
    select 5 as id, 'E' as ccode
)
select
    row_number() over (order by id desc) as new_id,
    ccode
from cte
order by id;
```
```sql
select *
from cte
with cte as (
    select 1 as id, 'A' as ccode
    union all select 2, 'B'
    union all select 3, 'C'
    union all select 4, 'D'
    union all select 5, 'E'
)
select 
    (select max(id) from cte) + 1 - id as id,
    ccode
from cte
order by ccode;
```
:::

---

## Задача от 04.09 19:00

:::{tip} Задача от 04.09 19:00
[ссылка](https://t.me/c/3825910988/777) \
Часто мы оцениваем активность клиентов в Т-0 (текущий месяц) и Т-1 (следующий месяц) от даты коммуникации.
В рамках задачи у вас есть 2 таблицы: коммуникации (id клиента, дата коммуникации) и активность (id клиента, месяц, флаг активен/не активен). 
На выходе у вас должна получиться таблица вида: id клиента, дата коммуникации, активность Т0 и активность Т1. На 1 клиента 1 строка.

Напишите запрос который позволяет решить эту задачу в ответ на это сообщение. Здесь вам придется немного забежать вперед и понять как делаются джоины)
:::

- Вопрос: как у нас на 1 клиента 1 строка, когда у одного клиента может быть две строки дата коммуникации в текущем и следующем месяце, нет?
- Евгений: имеется в виду, что это пилот с одной датой отправки коммуникации. Дата всегда в формате даты)) если нет, то вопросы к компетенции ДИ (Дата Инженера).

:::{note} Решения:
Евгений (ментор): в 99% случаев на интенсивах вижу, что многие извлекают номер месяца, а я наоборот привык приводить к первому или последнему дню месяца, как сделали вы (я именно про date_trunc):
```sql
-- в недельном интенсиве подобная задача 4 Практика 5
-- коммуникация пусть будет com а активность active (столбец act)
select c.id,
c.com_date -- это дата ком-ии
a0.act as t0,
a1.act as t1,
from com as c
left join active as a0 on c.id=a0.id and a0.month=date_trunc('month',c.com_date )
left join active as a1 on c.id=a1.id and a1.month=date_trunc('month',c.com_date ) + interval '1 month'
order by c.id asc
```

```sql
select
    c.id,
    c.date,
    a0.is_active as t0,
    a1.is_active as t1
from communications c
left join activity a0
    on c.id = a0.id
    and a0.month = date_trunc('month', c.date)
left join activity a1
    on c.id = a1.id
    and a1.month = date_trunc('month', c.date) + interval '1 month';
```

```sql
select
    c.client_id,
    c.communication_date, -- 1 клиент - 1 дата
    a0.is_active as t0, 
    a1.is_active as t1
from communications c
- - left join чтобы мы видели всех клиентов из communications
- - 1й джойн отвечает за т0 (активность в том же месяце, что и коммуникация)
left join activity a0
on c.client_id = a0.client_id
and a0.month = date_trunc('month', c.communication_date)   - - если месяц совпал берем значение is_active (тут допущение - считаем что месяц активности дата,  а не строка)
- - 2й джойн отвечает за т1 (активность в следующем месяце)
left join activity a1
on c.client_id = a1.client_id
- - если запись за месяц есть - берем is_active, а если нет - t1 будет null (потому что left join)
and a1.month = date_trunc('month', c.communication_date) + INTERVAL '1 month';
```

```sql
SELECT 
    c.id,
    c.com_date,
    a0.flag AS T_0,
    a1.flag AS T_1
FROM communication c
LEFT JOIN activity a0 
    ON c.id = a0.id 
   AND DATE_TRUNC('month', c.com_date) = a0.month
LEFT JOIN activity a1 
    ON c.id = a1.id 
   AND DATE_TRUNC('month', c.com_date) + INTERVAL '1 month' = a1.month;
```
:::

---

## Реал от 05.09

:::{tip} Реальная задача от 05.09
[ссылка](https://t.me/c/3825910988/831) \
Всем привет! Это не задачка, а повод для размышления))
Я в пятницу делал задачу:
У меня есть таблица с клиентами, есть таблица с просмотрами и кликами по баннерам, и есть таблица с продажами от этих кликов.
Проблема: таблицу просмотров и кликов перегрузили и часть кликов пропала, но у меня есть от них продажи.
Вопрос: как не потерять продажи?

P.S.: само собой таблицу с просмотрами и кликами нужно исправить, но это долго, а данные нужны были в пятницу.

- Клики уникальны, но в гранулярности клиент, плейсмент, код баннера и дата. Связь клика с покупкой лежит в отдельной витрине, где есть дата клика, плейсмент, код баннера и факт продажи. В пятницу нужны были все данные, просмотры, клики и продажи
- id вообще нет. Но там была проблема в том, что раньше клик был, а потом его не стало, поэтому продажу к нему больше не притянуть. Пришлось решать по другому
- Там проблема в том, что продажи нужно цеплять клик в клик по ключам: дата, клиент, плейсмент и код баннера. А в витрине с кликами и просмотрами этих данных нет, поэтому не прицепить так. Пришлось искать другой путь
:::

:::{note} Решение
Я придумал два варианта:
1. Умышленно замедлить клиентскую базу через кроссджоин на уникальные плейсменты, даты и продукты, а потом лефт джоином присоединить все. Но не получилось - выборка была слишком большой. 
2. Объединить их через Юнион. Где продажи, там столбцы с кликами и просмотрами = 0, где клики и просмотры, там продажи = 0. Потом юнион и агрегация.

- 2. Для юнион количество столбцов догоняем нулевыми значениями - это понятно. Но что нам даст сам юнион? - совсем непонятно) зачем их вообще соединять? Чтобы что-то посчитать группируя по какому-то полю? Т.е. по каждому клиенту можно вывести сразу и просмотры и продажи, включая нули?
- Ну после юнионов можно

- Я тоже не поняла, как именно формируется юнион во втором варианте. В таблице продаж есть поля плейсмент, баннер? Если нет, то как тогда эти данные притягивались к продажам?
- Да, конечно, в продажах есть  информация с какого баннера и плейсмента и в какую дату произошла продажа
:::

---

## Задача 05.09 17:00

:::{tip} Задача 05.09 17:00
[ссылка](https://t.me/c/3825910988/849) \
Вы анализируете клиентов, которые покупают подписку. Они подключают ее на триал период (пробный период, который длится 1 месяц), потом клиенты могут оплатить или не оплатить платную версию. Оплата списывается ровно через месяц после подключения триала, даже если клиент решил подключить платную версию раньше. 
Для решения задачи, у вас есть 3 таблицы:
Sales: client_id (1,2,3 и тд), deal_id (1,2,3 и тд), date (в формате 05.02.2026)
Clients: client_id (1,2,3 и тд), segment (1,2,3, где 1 малый бизнес, 2 - средний бизнес, 3 - крупный бизнес)
Payments: client_id (1,2,3 и тд), id (1,2,3 и тд), date (в формате 05.02.2026)

Вам нужно без использования where определить долю клиентов ММБ, которые оплатили подписку, подключенную в феврале.

В таблицах есть данные за 3 месяца: январь, февраль и март.
:::

:::{note} Решения

```sql
select
 round(
 -- числитель (те ммб, у которых оплата прошла через месяц)
    count(distinct case
          when extract(month from s.date) = 2 -- февраль
          and c.segment = 1 -- ММБ
          and p.client_id is not null -- p.client_id становится NULL из-за left join
          then s.client_id 
          end) * 100.0 /
 -- знаменатель (все ммб, подключившие подписку в феврале)
    -- + защита от деления на 0, если никто не купил
 nullif(count(distinct case
          when extract(month from s.date) = 2 -- февраль
          and c.segment = 1 -- ММБ
          then s.client_id 
          end), 0), 2) as conv_pct
from clients c
left join sales s
on c.client_id = s.client_id
left join payments p -- все с февральским триалом
on s.client_id = p.client_id
and p.date = (s.date + INTERVAL '1 month');
```

```sql
select
    count(distinct case
        when p.client_id is not null then s.client_id
    end)::decimal
    / count(distinct s.client_id) as share_paid
from sales s
join clients c
    on s.client_id = c.client_id
    and c.segment = 1
    and s.date >= '2026-02-01'
    and s.date < '2026-03-01'
left join payments p
    on s.client_id = p.client_id
    and p.date = s.date + interval '1 month';
```
:::

---

[ссылка](https://t.me/c/3825910988/873) от 06.09

Сегодня у вас объединение таблиц и агрегация. Сама по себе тема объединения довольно простая, но очень полезная. Расскажите, часто ли вы используете union/union all?

Агрегация - это еще один must have для аналитика. Без агрегации невозможно собрать ни один отчет. В этом блоке вы изучите базовые агрегатные функции, а также продвинутые (их кстати мало кто знает). Будьте внимательны с group by, один лишний столбец в group by и ваш результат не верный. 
Дополнительно, отправляю вам статью по агрегатным функциям: https://simulative.ru/blog/aggregate-functions-sql

Сегодня вечером будет две задачки)) в 17.00 и в 19.00

---

## Задача 06.09 17:00

:::{tip} Задача 06.09 17:00
[ссылка](https://t.me/c/3825910988/879) \
В рамках одного селекта нужно посчитать общее количество кликов, просмотров и уникальное количество кликов и просмотров, и CTR по месяцам.  При этом таблица выглядит вот так:
Дата (день в формате 19.03.2026), id клиента, событие (просмотр или клик), количество. Жду ваши ответы.
:::

:::{note} Решение
```sql
select
 date_trunc('month', event_date) as event_month,
    sum(case when event_type = 'просмотр' then qty else 0 end) as view_cnt,
    sum(case when event_type = 'клик' then qty else 0 end) as click_cnt,
 count(distinct case when event_type = 'просмотр' then client_id end) as unique_views,
    count(distinct case when event_type = 'клик' then client_id end) as unique_clicks,
    -- CTR = Количество кликов / Количество просмотров
    100.00 * sum(case when event_type = 'клик' then qty else 0 end) / nullif(sum(case when event_type = 'просмотр' then qty else 0 end), 0) as CTR
from events
group by date_trunc('month', event_date);
```

```sql
select
date_trunc('month', to_date(date, 'DD.MM.YYYY')) as month,
sum(case when event = 'клик' then quantity else 0 end) as total_clicks,
sum(case when event = 'просмотр' then quantity else 0 end) as total_views,
count(distinct case when event = 'клик' then client_id end) as unique_clicks,
count(distinct case when event = 'просмотр' then client_id end) as unique_views,
sum(case when event = 'клик' then quantity else 0 end) * 100.0 / 
nullif(sum(case when event = 'просмотр' then quantity else 0 end), 0) as ctr
group by date_trunc('month', to_date(date, 'DD.MM.YYYY'))
```
:::

---

## Задача 06.09 19:00

:::{tip} Задача 06.09 19:00
[ссылка](https://t.me/c/3825910988/885) \
Задача не очень обычная, но зато интересная. У нас есть две таблицы с клиентами: 
1. Control_group, с полями clinet_id, active_flg (в формате 1 и 0), segment (в формате 0-5, 5-10, 10-50, 50-500, 500-1000);
2. Target group, с такими же полями. 
В таблице с контрольной группой всего 15 клиентов. Вам нужно найти схожих клиентов из таблицы с целевой группой (там более 1 млн клиентов). 

Жду ваших решений)
:::

:::{note} Решение
[ссылка](https://t.me/c/3825910988/886) на размышление
[ссылка](https://t.me/c/3825910988/894) на подтвержденное решение:

```sql
select
c.client_id,
c.active_flg,
c.segment,
count(t.client_id) as similar_clients_count
from control_group c
left join target_group t
on c.active_flg = t.active_flg
and c.segment = t.segment
group by c.client_id, c.active_flg, c.segment
```

---
💡 В телеграм через контекстное меню можно выбрать на сообщении (задачи) "Посмотреть **х** ответов"
:::

---

## Зачача 07.09 17:00

:::{tip} Зачача 07.09 17:00
[ссылка](https://t.me/c/3825910988/909) \
В PostgreSQL у вас есть функция generate_series, но в других диалектах ее нет. Сможете сгенерировать календарь без нее?

Есть три основных способа) в разных диалектах причем разные и приходится разные знать. Но опция полезная, в том числе на собеседованиях спрашивают.
:::

:::{note} Решение
[ссылка](https://t.me/c/3825910988/915?thread=909) на решение. \
Или просмотром ответов на исходное [сообщение](https://t.me/c/3825910988/909)
:::

---

## Задача 07.09 19:00

:::{tip} Задача 07.09 19:00
У вас есть таблица с продуктом, количеством продаж и его стоимостью, соответственно, столбцы product, cnt и price. Вам нужно посчитать долю выручки каждого продукта от общей выручки в рублях, используя только оконные функции (без обычных агрегирующих) в рамках всего одного селекта.
:::

:::{note} Решение
```sql
select
 product,
        price,
        (price * cnt) as revenue,
        100.00 * (price * cnt) / sum(price * cnt) over() as share
from products;
```
:::

---